# Qwen2.5-Omni-3B 오디오 멀티모달 QLoRA 파인튜닝 (Google Colab)

이 노트북은 레포를 클론해 **환경설치 → 데이터 → 학습 → 추론 → 평가**를 순서대로 실행합니다.

**중요**: 메뉴 `런타임 > 런타임 유형 변경`에서 **하드웨어 가속기 = GPU** 로 설정하세요.
무료 T4(16GB)로 3B QLoRA 학습이 가능하나 느립니다.


## 1. GPU 확인
T4(Turing)는 bf16을 네이티브 지원하지 않으므로 이 노트북은 학습에 **fp16**을 사용합니다.


In [ ]:
!nvidia-smi
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    cap = torch.cuda.get_device_capability(0)
    print('GPU:', name, '| compute capability:', cap)
    # compute capability >= 8.0 (Ampere+) 이면 bf16 가능
    bf16_ok = cap[0] >= 8
    print('bf16 지원:', bf16_ok, '→ 학습 dtype:', 'bfloat16' if bf16_ok else 'float16')

## 2. 레포 클론


In [ ]:
!git clone --branch claude/local-multimodal-model-8mluhh https://github.com/minsik1313/minsik.git
%cd minsik
!ls -la

## 3. 의존성 설치
Colab에는 torch가 이미 설치되어 있으므로 ms-swift와 오디오/평가 패키지만 설치합니다.
(설치에 수 분 소요될 수 있습니다.)


In [ ]:
!pip install -q 'ms-swift[all]' librosa soundfile jiwer qwen-omni-utils
!swift --version

## 4. 데이터 준비 (합성 샘플)
smoke test용 합성 오디오 데이터를 생성합니다. 실데이터는 `scripts/prepare_data.py`의
`build_records_from_real_data()`를 구현한 뒤 `--synthesize` 없이 실행하세요.


In [ ]:
!python scripts/prepare_data.py --synthesize --n-per-class 12
!head -1 data/train.jsonl

## 5. 학습 (QLoRA)
T4 기준 **fp16 + MAX_LENGTH=1024**로 실행합니다.
Ampere 이상 GPU(A100 등)라면 아래 `TORCH_DTYPE`/`COMPUTE_DTYPE`를 `bfloat16`으로 바꾸세요.


In [ ]:
import torch
dtype = 'bfloat16' if (torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8) else 'float16'
print('학습 dtype =', dtype)
import os
os.environ['TORCH_DTYPE'] = dtype
os.environ['COMPUTE_DTYPE'] = dtype
os.environ['MAX_LENGTH'] = '1024'
os.environ['EPOCHS'] = '1'
!TORCH_DTYPE=$TORCH_DTYPE COMPUTE_DTYPE=$COMPUTE_DTYPE MAX_LENGTH=$MAX_LENGTH EPOCHS=$EPOCHS bash scripts/train.sh

## 6. 추론
학습된 어댑터로 val 셋을 배치 추론합니다.


In [ ]:
!bash scripts/infer.sh

## 7. 평가
합성 데이터는 분류형이라 accuracy로 평가합니다. ASR이라면 `--metric wer`.


In [ ]:
!python scripts/eval.py --metric accuracy

## 8. (선택) Google Drive에 결과 저장
Colab 세션이 끊기면 `outputs/`가 사라집니다. 어댑터/결과를 Drive에 백업하세요.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/minsik_outputs
!cp -r outputs/* /content/drive/MyDrive/minsik_outputs/ 2>/dev/null || echo '저장할 outputs 없음'
print('백업 완료: /content/drive/MyDrive/minsik_outputs')